# 06 — Shipping Analysis
Ship mode usage, average shipping days, and revenue/profit per shipping mode.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Revenue & Profit by Ship Mode ─────────────────────────────────────────────
ship = fetch_df(cur, """
    SELECT o.ship_mode,
           COUNT(DISTINCT o.order_id)  AS orders,
           ROUND(SUM(oi.sales),2)      AS revenue,
           ROUND(SUM(oi.profit),2)     AS profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100,2) AS margin_pct
    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.ship_mode ORDER BY revenue DESC
""")

bar_w = 0.35
x     = range(len(ship))
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar([i - bar_w/2 for i in x], ship["revenue"],
            bar_w, color=PALETTE[0], label="Revenue", zorder=3)
b2 = ax.bar([i + bar_w/2 for i in x], ship["profit"],
            bar_w, color=PALETTE[1], label="Profit", zorder=3)

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 500,
            fmt_usd(bar.get_height()),
            ha="center", va="bottom", fontsize=8, color=PALETTE[0])
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 100,
            fmt_usd(bar.get_height()),
            ha="center", va="bottom", fontsize=8, color=PALETTE[1])

ax.set_xticks(list(x))
ax.set_xticklabels(ship["ship_mode"], fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.legend(frameon=False)
ax.set_title("Revenue & Profit by Ship Mode")
ax.set_ylabel("USD")
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()


In [ ]:
# ── Avg Shipping Days per Mode ────────────────────────────────────────────────
days = fetch_df(cur, """
    SELECT ship_mode,
           ROUND(AVG(datediff(
               to_date(from_unixtime(unix_timestamp(ship_date,'M/d/yyyy'))),
               to_date(from_unixtime(unix_timestamp(order_date,'M/d/yyyy')))
           )),1) AS avg_ship_days,
           COUNT(DISTINCT order_id) AS orders
    FROM orders
    GROUP BY ship_mode ORDER BY avg_ship_days
""")

fig, ax = plt.subplots(figsize=(8, 4))
colors = [PALETTE[1] if d <= 3 else PALETTE[0] if d <= 5 else PALETTE[3]
          for d in days["avg_ship_days"][::-1]]
bars = ax.barh(days["ship_mode"][::-1], days["avg_ship_days"][::-1],
               color=colors, edgecolor="white")
for bar in bars:
    ax.text(bar.get_width() + 0.05,
            bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.1f} days",
            va="center", fontsize=10, fontweight="bold")

ax.set_xlabel("Average Days")
ax.set_title("Average Shipping Days by Ship Mode")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
